# ROS Bag 交互式分析 Notebook

这个notebook允许你：
1. 一次性读取ROS bag数据
2. 交互式地分析和可视化数据  
3. 点击热力图查看特征点时间分布
4. 无需重复读取bag文件

## 1. 导入必要的库

In [ ]:
import rosbag
import numpy as np
import matplotlib.pyplot as plt
from geometry_msgs.msg import Point
from scipy.interpolate import interp1d
import warnings
warnings.filterwarnings('ignore')

# 设置matplotlib为交互模式
%matplotlib widget

print("✓ 库导入成功")

## 2. 配置参数

In [ ]:
# 配置ROS bag文件路径
BAG_FILE = "experiments/vo_safe/test_simple_exp_2_210311.bag"
BAG_FILES = [
            #  "/home/smoggy/workspace_ros1/r2d2/experiments/fuel/test_simple_exp_2_182751.bag",
            #  "/home/smoggy/workspace_ros1/r2d2/experiments/vo_safe/test_simple_exp_2_182051.bag",
             "/home/smoggy/workspace_ros1/r2d2/experiments/fuel/test_simple_exp_1_182528.bag",
             "/home/smoggy/workspace_ros1/r2d2/experiments/vo_safe/test_simple_exp_1_181610.bag"
             ]
# BAG_FILE = "/home/smoggy/workspace_ros1/r2d2/experiments/vo_safe/test_simple_exp_3_210642.bag"
# BAG_FILE = "/home/smoggy/workspace_ros1/r2d2/experiments/fuel/test_simple_exp_3_212158.bag"
# BAG_FILE = "/home/smoggy/workspace_ros1/r2d2/experiments/fuel/test_simple_exp_4_212422.bag"

# 热力图参数
OV_WIDTH = 960
OV_HEIGHT = 540  
R2D2_WIDTH = 1280
R2D2_HEIGHT = 720
BIN_SIZE = 20

# 颜色范围
R2D2_SCALE = [0.0, 0.1]
OV_SCALE = [0.0, 1.0]

print(f"配置完成: {BAG_FILE}")

## 3. 读取ROS Bag数据（只需运行一次）

In [ ]:
exploration_datas = []
r2d2_pc_datas = []
ov_pc_datas = []
gt_odom_datas = []

for BAG_FILE in BAG_FILES:    
    print(f"正在读取ROS bag: {BAG_FILE}")

    exploration_data = []
    r2d2_pc_data = []
    ov_pc_data = []
    gt_odom_data = []

    with rosbag.Bag(BAG_FILE, 'r') as bag:
        for topic, msg, t in bag.read_messages():
            timestamp = t.to_sec()
            
            if topic == '/exploration_rate':
                rate = msg.data if hasattr(msg, 'data') else float(msg)
                exploration_data.append([timestamp, rate])
            elif topic == '/r2d2/visible_features_uv':
                r2d2_pc_data.append([timestamp, msg])
            elif topic == '/ov_msckf/loop_feats':
                ov_pc_data.append([timestamp, msg])
            elif topic == '/kingfisher/ground_truth/odometry':
                gt_odom_data.append([timestamp, msg])

    print(f"✓ 找到 {len(exploration_data)} 条探索率消息")
    print(f"✓ 找到 {len(r2d2_pc_data)} 条R2D2点云消息")
    print(f"✓ 找到 {len(ov_pc_data)} 条OV-MSCKF特征消息")
    print(f"✓ 找到 {len(gt_odom_data)} 条地面真实里程计消息")

    exploration_datas.append(exploration_data)
    r2d2_pc_datas.append(r2d2_pc_data)
    ov_pc_datas.append(ov_pc_data)
    gt_odom_datas.append(gt_odom_data)


## 4. 提取点云UV坐标和时间戳

In [ ]:
print("正在提取点云数据...")

# 为每个rosbag创建子图
num_bags = len(gt_odom_datas)
fig, axes = plt.subplots(num_bags, 2, figsize=(12, 4 * num_bags), squeeze=False)

all_starting_timestamps = []
all_durations = []

for idx, gt_odom_data in enumerate(gt_odom_datas):
    gt_timestamps = []
    gt_linear_vel = []
    gt_angular_vel = []
    starting_timestamp = None

    for timestamp, msg in gt_odom_data:
        gt_timestamps.append(timestamp)
        # Linear velocity components
        vx = msg.twist.twist.linear.x 
        vy = msg.twist.twist.linear.y
        vz = msg.twist.twist.linear.z
        linear_vel = np.sqrt(vx**2 + vy**2 + vz**2)  # Calculate magnitude
        if linear_vel > 0.1 and starting_timestamp is None:
            starting_timestamp = timestamp  # Relative to first timestamp

        # Angular velocity components
        wx = msg.twist.twist.angular.x
        wy = msg.twist.twist.angular.y
        wz = msg.twist.twist.angular.z
        angular_vel = np.sqrt(wx**2 + wy**2 + wz**2)  # Calculate magnitude

        gt_linear_vel.append(linear_vel)
        gt_angular_vel.append(angular_vel)

    # Convert to numpy arrays and relative time
    gt_timestamps = np.array(gt_timestamps)
    gt_linear_vel = np.array(gt_linear_vel)
    gt_angular_vel = np.array(gt_angular_vel)

    # Plot for this bag
    ax1 = axes[idx, 0]
    ax2 = axes[idx, 1]
    ax1.plot(gt_timestamps, gt_linear_vel, 'b-', label='Linear Velocity')
    ax1.grid(True)
    ax1.set_xlabel('Time (s)')
    ax1.set_ylabel('Linear Velocity (m/s)')
    ax1.legend()
    ax1.set_title(f'Bag {idx+1} Linear Velocity')

    ax2.plot(gt_timestamps, gt_angular_vel, 'r-', label='Angular Velocity')
    ax2.grid(True)
    ax2.set_xlabel('Time (s)')
    ax2.set_ylabel('Angular Velocity (rad/s)')
    ax2.legend()
    ax2.set_title(f'Bag {idx+1} Angular Velocity')

    if starting_timestamp is not None:
        print(f"[Bag {idx+1}] starting_timestamp:", starting_timestamp)
        print(f"[Bag {idx+1}] ending_timestamp:", gt_timestamps[-1])
        duration = gt_timestamps[-1] - starting_timestamp
        print(f"[Bag {idx+1}] ✓ 运动持续时间: {duration:.2f} 秒")
        all_starting_timestamps.append(starting_timestamp)
        all_durations.append(duration)
    else:
        print(f"[Bag {idx+1}] 未检测到速度大于0.1的起始时刻")
        all_starting_timestamps.append(None)
        all_durations.append(None)

plt.tight_layout()
plt.show()

In [ ]:
# 为每个rosbag创建子图，分别显示OV-MSCKF和R2D2特征热力图

import struct
import matplotlib.pyplot as plt

num_bags = len(r2d2_pc_datas) 
fig, axes = plt.subplots(num_bags, 2, figsize=(16, 4 * num_bags), squeeze=False)

for idx in range(num_bags):
    r2d2_pc_data = r2d2_pc_datas[idx]
    ov_pc_data = ov_pc_datas[idx]
    starting_timestamp = all_starting_timestamps[idx]
    duration = all_durations[idx]

    # 处理R2D2数据
    r2d2_u = []
    r2d2_v = []
    r2d2_intensity = []
    r2d2_time = []
    for timestamp, msg in r2d2_pc_data:
        msg_data = bytes(msg.data)
        point_step = msg.point_step
        num_points = msg.width
        for i in range(num_points):
            offset = i * point_step
            point_data = msg_data[offset:offset + point_step]
            u, v, intensity = struct.unpack('<fff', point_data)
            r2d2_u.append(u)
            r2d2_v.append(v)
            r2d2_intensity.append(intensity)
            r2d2_time.append(timestamp)

    # 处理OV-MSCKF数据
    ov_msckf_u_coords = []
    ov_msckf_v_coords = []
    ov_time = []
    for timestamp, msg in ov_pc_data:
        for i in range(len(msg.channels)):
            u = msg.channels[i].values[2]
            v = msg.channels[i].values[3]
            ov_msckf_u_coords.append(u)
            ov_msckf_v_coords.append(v)
            ov_time.append(timestamp)

    # 转换为numpy数组
    r2d2_u = np.array(r2d2_u)
    r2d2_v = np.array(r2d2_v)
    r2d2_intensity = np.array(r2d2_intensity)
    r2d2_time = np.array(r2d2_time)
    ov_u = np.array(ov_msckf_u_coords)
    ov_v = np.array(ov_msckf_v_coords)
    ov_time = np.array(ov_time)

    print(f"[Bag {idx+1}] ✓ 提取了 {len(r2d2_u)} 个R2D2特征点")
    print(f"[Bag {idx+1}] ✓ 提取了 {len(ov_u)} 个OV-MSCKF特征点")
    if len(r2d2_u) > 0:
        print(f"  R2D2 UV范围: u=[{r2d2_u.min():.1f}, {r2d2_u.max():.1f}], v=[{r2d2_v.min():.1f}, {r2d2_v.max():.1f}]")

    # OV-MSCKF热力图
    ax_ov = axes[idx, 0]
    if len(ov_u) > 0 and duration is not None and duration > 0:
        heatmap_ov, xedges_ov, yedges_ov = np.histogram2d(
            ov_u, ov_v,
            bins=[OV_WIDTH // BIN_SIZE, OV_HEIGHT // BIN_SIZE],
            range=[[0, OV_WIDTH], [0, OV_HEIGHT]]
        )
        heatmap_ov = heatmap_ov / duration
        im_ov = ax_ov.imshow(
            heatmap_ov.T,
            origin='lower',
            extent=[xedges_ov[0], xedges_ov[-1], yedges_ov[0], yedges_ov[-1]],
            aspect='auto',
            cmap='viridis',
            vmin=0,
            vmax=0.9
        )
        plt.colorbar(im_ov, ax=ax_ov, label='OV-MSCKF feature density (points/s)')
    ax_ov.set_title(f'Bag {idx+1} OV-MSCKF feature heatmap')
    ax_ov.set_xlabel('u (pixels)')
    ax_ov.set_ylabel('v (pixels)')
    ax_ov.grid(False)

    # R2D2热力图
    ax_r2d2 = axes[idx, 1]
    if len(r2d2_u) > 0 and duration is not None and duration > 0:
        heatmap_r2d2, xedges_r2d2, yedges_r2d2 = np.histogram2d(
            r2d2_u, r2d2_v,
            bins=[R2D2_WIDTH // BIN_SIZE, R2D2_HEIGHT // BIN_SIZE],
            range=[[0, R2D2_WIDTH], [0, R2D2_HEIGHT]],
            weights=r2d2_intensity
        )
        heatmap_r2d2 = heatmap_r2d2 / duration
        im_r2d2 = ax_r2d2.imshow(
            heatmap_r2d2.T,
            origin='lower',
            extent=[xedges_r2d2[0], xedges_r2d2[-1], yedges_r2d2[0], yedges_r2d2[-1]],
            aspect='auto',
            cmap='viridis',
            vmin=0,
            vmax=0.1
        )
        plt.colorbar(im_r2d2, ax=ax_r2d2, label='r2d2 feature density (points/s)')
    ax_r2d2.set_title(f'Bag {idx+1} r2d2 feature heatmap')
    ax_r2d2.set_xlabel('u (pixels)')
    ax_r2d2.set_ylabel('v (pixels)')
    ax_r2d2.grid(False)

plt.tight_layout()
plt.show()